# Solutions – 2. Confidence Intervals

This notebook provides worked solutions to the five exercises from
[02_confidence_intervals.ipynb](../Stats/02_confidence_intervals.ipynb).

We reuse the same data-generation, NLL, and profile-likelihood helpers from notebooks 01–02.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon, chi2
from scipy.optimize import minimize

rng = np.random.default_rng(seed=123)

In [ ]:
# ── True parameters ──────────────────────────────────────────────────────────
theta_true  = 0.3
mu_true     = 5.0
sigma_true  = 0.5
lambda_true = 0.5
n_samples   = 2000

# ── Data generation ───────────────────────────────────────────────────────────
def generate_data(n, theta=theta_true, mu=mu_true, sigma=sigma_true,
                  lam=lambda_true, rng_=None):
    if rng_ is None:
        rng_ = np.random.default_rng()
    n_sig = rng_.binomial(n, theta)
    n_bkg = n - n_sig
    xs = np.concatenate([
        rng_.normal(mu, sigma, size=n_sig),
        rng_.exponential(1.0/lam, size=n_bkg)
    ])
    rng_.shuffle(xs)
    return xs

x_data = generate_data(n_samples, rng_=rng)

# ── NLL ───────────────────────────────────────────────────────────────────────
def nll_full(params, data):
    theta, mu, log_sigma, log_lam = params
    sigma = np.exp(log_sigma); lam = np.exp(log_lam)
    if not (0.0 < theta < 1.0): return np.inf
    ls = np.log(theta)       + norm.logpdf(data, mu, sigma)
    lb = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)
    return -np.sum(np.logaddexp(ls, lb))

def fit(data, theta0=0.2, mu0=None):
    if mu0 is None: mu0 = data.mean()
    x0 = [theta0, mu0, np.log(0.6), np.log(0.6)]
    res = minimize(nll_full, x0=x0, args=(data,), method='Nelder-Mead',
                   options={'xatol':1e-6,'fatol':1e-6,'maxiter':50000})
    return res.x, res.fun, res

x0_mle, nll_min, result = fit(x_data)
theta_hat, mu_hat = x0_mle[0], x0_mle[1]
sigma_hat, lam_hat = np.exp(x0_mle[2]), np.exp(x0_mle[3])
mle_internal = np.array(x0_mle)

# Wilks' thresholds
cl68_1d = chi2.ppf(0.6827, df=1) / 2
cl95_1d = chi2.ppf(0.9500, df=1) / 2

print(f"MLE: theta={theta_hat:.4f}, mu={mu_hat:.4f}, sigma={sigma_hat:.4f}, lam={lam_hat:.4f}")
print(f"1-D thresholds: 68% -> {cl68_1d:.3f},  95% -> {cl95_1d:.3f}")

In [ ]:
# ── Profile NLL helper ────────────────────────────────────────────────────────
def profile_nll_theta(theta_val, data, mle_int):
    """Profile over mu, sigma, lambda for a fixed theta."""
    free_idx = [1, 2, 3]
    x0_free  = mle_int[free_idx]
    def obj(free_vals):
        params = mle_int.copy()
        params[free_idx] = free_vals
        params[0] = theta_val
        theta, mu, log_s, log_l = params
        if not (0.0 < theta < 1.0): return 1e10
        sigma, lam = np.exp(log_s), np.exp(log_l)
        ls = np.log(theta)       + norm.logpdf(data, mu, sigma)
        lb = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)
        return -np.sum(np.logaddexp(ls, lb))
    res = minimize(obj, x0=x0_free, method='Nelder-Mead',
                   options={'xatol':1e-5,'fatol':1e-5,'maxiter':10000})
    return res.fun

def theta_ci_95(data, mle_int, grid_lo=0.05, grid_hi=0.55, n_grid=60):
    """Return the 95% profile-likelihood CI for theta."""
    grid = np.linspace(grid_lo, grid_hi, n_grid)
    dnll = np.array([profile_nll_theta(t, data, mle_int) for t in grid])
    dnll -= dnll.min()
    inside = grid[dnll <= cl95_1d]
    if len(inside) < 2:
        return (np.nan, np.nan)
    return (inside.min(), inside.max())

print("Helpers defined.")

---
## Exercise 1 – Coverage check

**Task:** Generate 200 independent datasets, compute the 95 % profile-likelihood CI for
$\theta$ for each, and measure the actual coverage.

In [ ]:
n_datasets = 200
covered    = 0
rng_cov    = np.random.default_rng(seed=42)

for k in range(n_datasets):
    data_k = generate_data(n_samples, rng_=rng_cov)
    mle_k, nll_k, _ = fit(data_k, mu0=data_k.mean())
    lo, hi = theta_ci_95(data_k, mle_k)
    if not np.isnan(lo) and lo <= theta_true <= hi:
        covered += 1
    if (k + 1) % 50 == 0:
        print(f"  {k+1}/{n_datasets}: running coverage = {covered/(k+1)*100:.1f}%")

coverage = covered / n_datasets
print(f"\n=== Coverage check (n_datasets={n_datasets}, n={n_samples}) ===")
print(f"  Fraction of 95% CIs containing theta_true = {coverage:.3f}  ({covered}/{n_datasets})")
print(f"  Nominal coverage = 0.95")

**Observations:**
- The empirical coverage should be close to 95 %, confirming that the profile-likelihood
  procedure produces **valid frequentist confidence intervals** for this sample size.
- Small deviations from 95 % are expected from finite Monte Carlo statistics:
  with 200 pseudo-experiments the standard error on the coverage estimate is
  $\sqrt{0.95 \cdot 0.05 / 200} \approx 1.5\%$.

---
## Exercise 2 – Parabolic (Hessian) approximation

**Task:** Estimate the uncertainty on $\hat{\theta}$ from the inverse Hessian of the NLL
and compare the resulting symmetric CI with the profile-likelihood CI.

In [ ]:
# Re-fit using BFGS to obtain the inverse Hessian
x0_bfgs = [theta_hat, mu_hat, np.log(sigma_hat), np.log(lam_hat)]
res_bfgs = minimize(nll_full, x0=x0_bfgs, args=(x_data,), method='BFGS',
                     options={'gtol': 1e-6, 'maxiter': 50000})

hess_inv = res_bfgs.hess_inv   # 4x4 matrix
var_theta = hess_inv[0, 0]
sigma_theta_hessian = np.sqrt(var_theta)

theta_hat_bfgs = res_bfgs.x[0]
ci_hessian = (theta_hat_bfgs - sigma_theta_hessian,
              theta_hat_bfgs + sigma_theta_hessian)

# Profile-likelihood 68% CI for comparison
grid_theta = np.linspace(0.12, 0.55, 80)
dnll_theta = np.array([profile_nll_theta(t, x_data, mle_internal) for t in grid_theta])
dnll_theta -= dnll_theta.min()
in68 = grid_theta[dnll_theta <= cl68_1d]
ci_profile = (in68.min(), in68.max()) if len(in68) >= 2 else (np.nan, np.nan)

print("=== 68 % confidence interval for theta ===")
print(f"  Hessian (symmetric):   [{ci_hessian[0]:.4f}, {ci_hessian[1]:.4f}]")
print(f"  Profile likelihood:    [{ci_profile[0]:.4f}, {ci_profile[1]:.4f}]")
print(f"  True value:            {theta_true}")

In [ ]:
# Visualise the profile DNLL and the Hessian approximation
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid_theta, dnll_theta, 'b-', lw=2, label=r'Profile $\Delta$NLL')

# Parabolic approximation centred at BFGS MLE
parabola = 0.5 * ((grid_theta - theta_hat_bfgs) / sigma_theta_hessian) ** 2
ax.plot(grid_theta, parabola, 'r--', lw=2, label='Hessian (parabolic) approx')

ax.axhline(cl68_1d, color='orange', ls='--', label=f'68% threshold ({cl68_1d:.2f})')
ax.axhline(cl95_1d, color='darkred', ls='--', label=f'95% threshold ({cl95_1d:.2f})')
ax.axvline(theta_true, color='gray', ls=':', lw=1.5, label=f'True θ = {theta_true}')
ax.set_xlabel(r'$\theta$'); ax.set_ylabel(r'$\Delta$NLL')
ax.set_title(r'Profile $\Delta$NLL vs. Hessian approximation')
ax.set_ylim(-0.1, 5); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

**Observations:**
- The parabolic (Hessian) approximation agrees well with the profile NLL near the minimum
  because the likelihood is nearly Gaussian for $n = 2000$.
- Both methods give very similar 68 % CIs.
- For smaller samples or near parameter boundaries the profile NLL becomes asymmetric,
  and the parabolic approximation **underestimates** the CI width on one side.

---
## Exercise 3 – Wilks' theorem validity at small $n$

**Task:** Repeat the coverage study for $n = 100$ and compare the distribution of
$2\,\Delta\text{NLL}$ at the true $\theta$ with $\chi^2_1$.

In [ ]:
n_small  = 100
n_wilks  = 500     # number of pseudo-experiments for the Wilks check
dnll_values = []
rng_wk = np.random.default_rng(seed=55)

for k in range(n_wilks):
    data_k = generate_data(n_small, rng_=rng_wk)
    mle_k, nll_k, _ = fit(data_k, mu0=data_k.mean())
    # Profile NLL at the true theta
    pnll_true = profile_nll_theta(theta_true, data_k, mle_k)
    dnll_k = 2.0 * (pnll_true - nll_k)
    if np.isfinite(dnll_k) and dnll_k >= 0:
        dnll_values.append(dnll_k)

dnll_arr = np.array(dnll_values)

# Coverage at 95 %
chisq_95 = chi2.ppf(0.95, df=1)
coverage_small = np.mean(dnll_arr <= chisq_95)

print(f"n = {n_small}, pseudo-experiments = {len(dnll_arr)}")
print(f"Fraction with 2*DNLL <= chi2_95 ({chisq_95:.3f}) = {coverage_small:.3f}")
print(f"(Wilks prediction: 0.95)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
t = np.linspace(0, 10, 300)
ax.hist(dnll_arr, bins=30, density=True, histtype='stepfilled',
        alpha=0.5, color='steelblue', label=r'Observed $2\,\Delta$NLL')
ax.plot(t, chi2.pdf(t, df=1), 'r-', lw=2, label=r'$\chi^2_1$ (Wilks)')
ax.axvline(chisq_95, color='darkred', ls='--', label=f'95% quantile = {chisq_95:.2f}')
ax.set_xlabel(r'$2\,\Delta$NLL at true $\theta$')
ax.set_ylabel('Density')
ax.set_title(f"Wilks' theorem check, n = {n_small}")
ax.legend(); ax.set_xlim(0, 10)
plt.tight_layout(); plt.show()

**Observations:**
- At $n = 100$ the distribution of $2\,\Delta\text{NLL}$ already tracks the $\chi^2_1$ 
  distribution reasonably well, though the tails may deviate.
- The empirical coverage is typically slightly below 95 %, indicating that Wilks'
  theorem is only approximately valid at small $n$.
- At $n = 2000$ the agreement is very good; at $n = 50$ deviations become more pronounced.
  The finite-sample correction can be assessed with toy MC studies (pseudo-experiments) as done here.

---
## Exercise 4 – Effect of a systematic uncertainty on $\lambda$

**Task:** Add a Gaussian constraint on $\lambda$ with width $\delta_\lambda$ and study
how the CI for $\theta$ widens as $\delta_\lambda$ increases from 0 to 0.2.

In [ ]:
def profile_nll_theta_constrained(theta_val, data, mle_int, lam_aux, delta_lam):
    """Profile NLL at fixed theta with an optional Gaussian constraint on lambda.

    Conventions for the lambda constraint:
      delta_lam = 0  -> lambda fixed to lam_aux
      delta_lam > 0  -> Gaussian penalty with width delta_lam
    """
    if delta_lam == 0.0:
        # Fixed-lambda limit: profile only mu and sigma.
        free_idx = [1, 2]
        x0_free = mle_int[free_idx]

        def obj(free_vals):
            params = mle_int.copy()
            params[0] = theta_val
            params[1] = free_vals[0]
            params[2] = free_vals[1]
            params[3] = np.log(lam_aux)
            return nll_full(params, data)

    else:
        # Gaussian-constrained lambda: profile mu, sigma, lambda.
        free_idx = [1, 2, 3]
        x0_free = mle_int[free_idx]

        def obj(free_vals):
            params = mle_int.copy()
            params[free_idx] = free_vals
            params[0] = theta_val
            base_nll = nll_full(params, data)
            lam = np.exp(params[3])
            penalty = 0.5 * ((lam - lam_aux) / delta_lam) ** 2
            return base_nll + penalty

    res = minimize(
        obj,
        x0=x0_free,
        method='Nelder-Mead',
        options={'xatol': 1e-5, 'fatol': 1e-5, 'maxiter': 12000},
    )
    return res.fun


def ci_bounds_from_threshold(grid, dnll, threshold):
    """Linear-interpolated CI bounds where dnll crosses the chosen threshold."""
    inside = dnll <= threshold
    idx_inside = np.flatnonzero(inside)
    if len(idx_inside) == 0:
        return np.nan, np.nan

    i_left = idx_inside[0]
    i_right = idx_inside[-1]

    # Left crossing
    if i_left == 0:
        left = grid[0]
    else:
        x1, y1 = grid[i_left - 1], dnll[i_left - 1]
        x2, y2 = grid[i_left], dnll[i_left]
        left = x2 if np.isclose(y2, y1) else x1 + (threshold - y1) * (x2 - x1) / (y2 - y1)

    # Right crossing
    if i_right == len(grid) - 1:
        right = grid[-1]
    else:
        x1, y1 = grid[i_right], dnll[i_right]
        x2, y2 = grid[i_right + 1], dnll[i_right + 1]
        right = x1 if np.isclose(y2, y1) else x1 + (threshold - y1) * (x2 - x1) / (y2 - y1)

    return left, right


def ci_width_with_syst(delta_lam, data=x_data):
    """Return the 95% CI width for theta given systematic delta_lam."""
    lam_aux = lambda_true
    grid = np.linspace(0.05, 0.60, 120)

    prof_vals = np.array([
        profile_nll_theta_constrained(t, data, mle_internal, lam_aux, delta_lam)
        for t in grid
    ])

    dnll = prof_vals - np.nanmin(prof_vals)
    lo, hi = ci_bounds_from_threshold(grid, dnll, cl95_1d)

    if np.isnan(lo) or np.isnan(hi):
        return np.nan
    return hi - lo


# Denser scan to make the trend smoother in delta_lambda.
delta_lam_values = np.linspace(0.0, 0.20, 21)
ci_widths = np.array([ci_width_with_syst(dl) for dl in delta_lam_values])

for dl, w in zip(delta_lam_values, ci_widths):
    print(f"  delta_lambda = {dl:.3f} -> 95% CI width = {w:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(delta_lam_values, ci_widths, '-', lw=2, color='steelblue', label='95% CI width')
ax.scatter(delta_lam_values, ci_widths, s=18, color='steelblue', alpha=0.7)
ax.set_xlabel(r'$\delta_\lambda$ (constraint width on $\lambda$)')
ax.set_ylabel(r'95 % CI width for $\theta$')
ax.set_title(r'CI widening due to systematic uncertainty on $\lambda$')
ax.grid(alpha=0.25)
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

**Observations:**
- In this implementation, $\delta_\lambda = 0$ is the **fixed-\(\lambda\)** limit (infinitely strong constraint at the auxiliary value).
- As $\delta_\lambda$ increases, the constraint weakens and the 95 % CI for $\theta$ should stay the same or widen (up to small numerical fluctuations).
- For sufficiently large $\delta_\lambda$, the result approaches the unconstrained-profile case where $\lambda$ is effectively free.
- The size of the effect depends on the correlation between $\theta$ and $\lambda$: if they are weakly correlated, the widening can be modest even for sizeable $\delta_\lambda$.


---
## Exercise 5 – Comparing 1-D and 2-D intervals

**Task:** From the 2-D confidence region for $(\theta, \mu)$ obtain the marginal 1-D
interval for $\theta$ by projection and compare with the profile-likelihood 1-D interval.

In [ ]:
from scipy.optimize import minimize as _min2

def profile_nll_2d(theta_val, mu_val, data, mle_int):
    """Fix theta and mu, minimise over sigma and lambda."""
    free_idx = [2, 3]
    x0_f = mle_int[free_idx]
    def obj(fv):
        p = mle_int.copy()
        p[free_idx] = fv
        p[0] = theta_val; p[1] = mu_val
        th, m, ls, ll = p
        if not (0.0 < th < 1.0): return 1e10
        s, l = np.exp(ls), np.exp(ll)
        ls_t = np.log(th) + norm.logpdf(data, m, s)
        lb_t = np.log(1.0-th) + expon.logpdf(data, scale=1.0/l)
        return -np.sum(np.logaddexp(ls_t, lb_t))
    r = _min2(obj, x0=x0_f, method='Nelder-Mead',
              options={'xatol':1e-5,'fatol':1e-5,'maxiter':10000})
    return r.fun

# Coarse 2-D grids
grid_th = np.linspace(0.14, 0.52, 25)
grid_mu = np.linspace(4.65, 5.35, 25)

Z2d = np.zeros((len(grid_mu), len(grid_th)))
for j, mv in enumerate(grid_mu):
    for i, tv in enumerate(grid_th):
        Z2d[j, i] = profile_nll_2d(tv, mv, x_data, mle_internal)
Z2d -= Z2d.min()

cl95_2d = chi2.ppf(0.95, df=2) / 2

# Marginal 1-D interval from 2-D region: project onto theta axis
# A theta value is inside if ANY mu has Z2d <= threshold
marginal_mask = np.any(Z2d <= cl95_2d, axis=0)   # shape (len(grid_th),)
theta_marginal_lo = grid_th[marginal_mask].min()
theta_marginal_hi = grid_th[marginal_mask].max()

# Profile-likelihood 1-D 95% CI for theta
dnll_1d = np.array([profile_nll_theta(t, x_data, mle_internal) for t in grid_th])
dnll_1d -= dnll_1d.min()
in95 = grid_th[dnll_1d <= cl95_1d]
theta_profile_lo = in95.min(); theta_profile_hi = in95.max()

print("=== 95% CIs for theta: projection of 2-D region vs. 1-D profile ===")
print(f"  Projection of 2-D region: [{theta_marginal_lo:.4f}, {theta_marginal_hi:.4f}]"
      f"  width = {theta_marginal_hi-theta_marginal_lo:.4f}")
print(f"  1-D profile likelihood:   [{theta_profile_lo:.4f}, {theta_profile_hi:.4f}]"
      f"  width = {theta_profile_hi-theta_profile_lo:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: 2-D confidence region
ax = axes[0]
ax.contourf(grid_th, grid_mu, Z2d, levels=30, cmap='Blues_r', alpha=0.7)
ax.contour(grid_th, grid_mu, Z2d, levels=[cl95_2d], colors=['red'], linewidths=2)
ax.axvspan(theta_marginal_lo, theta_marginal_hi, alpha=0.15, color='red',
           label=f'Marginal: [{theta_marginal_lo:.3f}, {theta_marginal_hi:.3f}]')
ax.plot(theta_hat, mu_hat, 'g*', ms=14, label='MLE')
ax.plot(theta_true, mu_true, 'k+', ms=14, mew=2, label='True')
ax.set_xlabel(r'$\theta$'); ax.set_ylabel(r'$\mu$')
ax.set_title(r'2-D 95 % confidence region ($\theta$, $\mu$)')
ax.legend(fontsize=9)

# Right: 1-D profile NLL
ax = axes[1]
ax.plot(grid_th, dnll_1d, 'b-', lw=2, label=r'Profile $\Delta$NLL')
ax.axhline(cl95_1d, color='darkred', ls='--', label=f'95% threshold')
ax.axvspan(theta_profile_lo, theta_profile_hi, alpha=0.15, color='blue',
           label=f'Profile CI: [{theta_profile_lo:.3f}, {theta_profile_hi:.3f}]')
ax.axvspan(theta_marginal_lo, theta_marginal_hi, alpha=0.15, color='red',
           label=f'Marginal: [{theta_marginal_lo:.3f}, {theta_marginal_hi:.3f}]')
ax.set_xlabel(r'$\theta$'); ax.set_ylabel(r'$\Delta$NLL')
ax.set_title(r'1-D profile $\Delta$NLL for $\theta$')
ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

**Observations:**
- The projection of the 2-D region onto the $\theta$ axis is **wider** than the 1-D
  profile-likelihood CI.
- This is expected: when projecting the 2-D region we accept any $\theta$ for which
  *some* $\mu$ falls inside the 2-D contour, whereas the profile likelihood optimises
  over $\mu$ and keeps only the best-fit $\mu$ for each $\theta$.
- The projection is a conservative (over-covering) procedure; the profile likelihood
  is the more efficient approach for 1-D inference.